In [3]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [227]:
import pandas as pd
from datetime import datetime
from collections import defaultdict

# Define time period
start = "2025-04-01T00:00:00Z"  # adjust as needed

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org', 'HTOC Comm', 'HTOC legacy data']
final_results = []
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=threatAssess,associatedGroups,tags,observations&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")

observed_src


Querying owner: HTOC Org

Querying owner: HTOC Comm

Querying owner: HTOC legacy data

Retrieved 8558 unique observed indicators.


,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,source,md5,sha1,sha256,size,text,url,Subject,address,indicator
0,4697066,2024-06-05T13:07:49Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-07T11:39:54Z,5.0,99,5.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.19.210.34
1,4883604,2024-09-09T18:29:05Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-05-07T11:39:50Z,3.0,54,3.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,clickme.thryv.com
2,6755399444006876,2025-04-23T15:00:09Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-07T11:39:07Z,5.0,99,5.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,169.150.203.16
3,3726701,2021-04-27T11:57:47Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-07T11:39:07Z,3.0,50,0.68,...,LookingGlass Collection,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.230.63.171
4,5263019,2025-01-22T16:32:25Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-05-07T11:39:06Z,2.0,67,1.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,facebookmail.com
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,6755399446008706,2025-05-05T18:05:18Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2025-05-05T18:05:37Z,3.0,0,3.00,...,NaN,NaN,NaN,NaN,NaN,tcp://freeaway.wikaba.com:389,NaN,NaN,NaN,tcp://freeaway.wikaba.com:389
9996,6755399446008705,2025-05-05T18:05:18Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2025-05-05T18:05:36Z,3.0,0,4.00,...,NaN,NaN,NaN,NaN,NaN,tcp://frameworksc04.quadrantbd.com:443,NaN,NaN,NaN,tcp://frameworksc04.quadrantbd.com:443
9997,6755399446008704,2025-05-05T18:05:18Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2025-05-05T18:05:36Z,3.0,0,3.00,...,NaN,NaN,NaN,NaN,NaN,tcp://emailcrypt.mobwork.net:443,NaN,NaN,NaN,tcp://emailcrypt.mobwork.net:443
9998,6755399446008703,2025-05-05T18:05:18Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2025-05-05T18:05:36Z,3.0,0,3.00,...,NaN,NaN,NaN,NaN,NaN,tcp://dy.skypetw.com:443,NaN,NaN,NaN,tcp://dy.skypetw.com:443


In [210]:
observed_src.to_csv(r'C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\apiData.csv')

In [209]:
observed_src[observed_src['summary'] == '45.84.107.128']

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,whoisActive,sha256,md5,sha1,size,url,Subject,text,address,indicator
45,5629499539130854,2025-04-25T13:02:39Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-06T11:32:11Z,4.0,71.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.84.107.128


In [228]:
import pandas as pd
from datetime import timedelta

# Initialize an empty DataFrame to store filtered tags
filtered_tags = pd.DataFrame()

# Loop through each row in observed_src
for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    
    if isinstance(tags_data, list):
        # Normalize the tags data for the current row
        tags = pd.json_normalize(tags_data)

        # Ensure 'name' column is of string type
        tags['name'] = tags['name'].astype(str)

        # Create a new column with a list of all tag names
        all_tags_list = tags['name'].tolist()

        # Filter for "API" tags
        api_tags = tags[tags['name'].str.contains('API', case=False, na=False)].copy()

        if not api_tags.empty:
            # Add metadata columns and all_tags list as a single value (not as a series)
            for col in ['summary', 'observations', 'description', 'type', 'dateAdded', 'lastModified', 'lastObserved']:
                api_tags[col] = row.get(col)
            
            # Assign the all_tags list as a single value for the entire set of API tags
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)

            # Append to the filtered DataFrame
            filtered_tags = pd.concat([filtered_tags, api_tags], ignore_index=True)

# Ensure 'lastUsed' is datetime
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')

# Define cutoff time in UTC
cutoff = pd.Timestamp.utcnow()

# Filter to last 48 hours
recent_tags = filtered_tags[filtered_tags['lastObserved'] >= cutoff - timedelta(hours=48)].copy()

# Extract partner names
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

# Group partners per summary
partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))]).reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)

# Merge partner information back into recent_tags
recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')

# Apply additional filters
recent_tags = recent_tags[recent_tags['partner_count'] >= 2]
recent_tags = recent_tags[recent_tags['observations'] < 15000]
recent_tags['dateAdded'] = pd.to_datetime(recent_tags['dateAdded'], errors='coerce')

# Optional: Filter by dateAdded in the last 30 days
date_added_cutoff = pd.Timestamp.now(tz='UTC') - timedelta(days=30)
# recent_tags = recent_tags[recent_tags['dateAdded'] >= date_added_cutoff]

# Drop duplicates based on 'summary'
recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

# Drop unnecessary columns
columns_to_drop = ['techniqueId', 'tactics.data', 'tactics.count', 'platforms.data', 'platforms.count', 'partner', 'description']
recent_tags = recent_tags.drop(columns=[col for col in columns_to_drop if col in recent_tags.columns], errors='ignore')

#  Remove rows where 'all_tags' contains 'I&W'
recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: 'I&W' in x)]

#date added no later than 30 days
recent_tags = recent_tags[recent_tags['dateAdded'] >= date_added_cutoff]

recent_tags


,id,name,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,all_tags,partner_count,partners
305,471298,DHA Splunk API,2025-05-07T11:31:33Z,190.92.174.130,1642,Address,2025-04-11 17:46:48+00:00,2025-05-07T11:33:41Z,2025-05-06 00:00:00+00:00,"[DHA Splunk API, APT & Targeted Attacks, unaut...",2,"DHA, NIH"
579,471298,DHA Splunk API,2025-05-07T11:31:33Z,190.92.174.30,41,Address,2025-04-11 17:46:48+00:00,2025-05-06T07:36:43Z,2025-05-07 00:00:00+00:00,"[DHA Splunk API, APT & Targeted Attacks, unaut...",2,"CDC, DHA"
805,471298,DHA Splunk API,2025-05-07T11:31:33Z,190.92.174.36,10216,Address,2025-04-11 17:46:48+00:00,2025-05-06T06:48:53Z,2025-05-06 00:00:00+00:00,"[DHA Splunk API, APT & Targeted Attacks, unaut...",3,"CDC, DHA, NIH"


In [235]:
import os
import json
import time
import urllib.parse
import urllib3
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords

#VT_API_KEY - a8b3e24dbd2e6c0cb002784aeb8fee6217a6a425cb11ddf9a3d3361281fbbb08
#OTX_API_KEY - ea83cf4792fc5db7acc741e82286c0b717fc99be98ec5b14de7e63cd3f74bcfe
# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Ensure NLTK stopwords are downloaded
nltk.download('stopwords')
STOP_WORDS = set(stopwords.words('english'))

# === CONFIG ===
FILE_PATH = r'C:\Users\jaskew\Documents\project_repository\data\processed\BingSearchData.json'
VT_API_KEY = "a8b3e24dbd2e6c0cb002784aeb8fee6217a6a425cb11ddf9a3d3361281fbbb08"
OTX_API_KEY = "ea83cf4792fc5db7acc741e82286c0b717fc99be98ec5b14de7e63cd3f74bcfe"


# === Helper Functions ===

def load_saved_links(file_path):
    if not os.path.exists(file_path):
        return set()
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return {entry['link'] for entry in data if 'link' in entry}
    except (json.JSONDecodeError, IOError):
        return set()

def fetch_virustotal_data(ip):
    url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"
    headers = {"x-apikey": VT_API_KEY}
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error querying VirusTotal API for {ip}: {e}")
        return None

def fetch_otx_data(ip):
    url = f"https://otx.alienvault.io/api/v1/indicators/IPv4/{ip}/general"
    headers = {"X-OTX-API-KEY": OTX_API_KEY}
    
    try:
        # First try with the correct domain
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.SSLError as ssl_err:
        print(f"SSL Error occurred: {ssl_err}")
        # If SSL verification fails, we can try with verify=False (for testing only)
        print("Attempting request with SSL verification disabled (not recommended for production)...")
        response = requests.get(url, headers=headers, verify=False)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error querying OTX API for {ip}: {e}")
        return None

def save_to_json(entry, file_path):
    try:
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    data = json.load(f)
                except json.JSONDecodeError:
                    data = []
        else:
            data = []

        data.append(entry)
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
    except IOError as e:
        print(f"Error saving data: {e}")

# === Main Script ===

def main():
    # Ensure 'summary' column exists
    if 'summary' not in recent_tags.columns:
        print("The 'summary' column is missing in the provided DataFrame.")
        return
    # Extract search terms from 'summary' column
    search_terms = recent_tags['summary'].dropna().unique().tolist()
    print(f"Found {len(search_terms)} unique search terms in 'summary' column.")

    saved_links = load_saved_links(FILE_PATH)
    print(f"Loaded {len(saved_links)} previously saved links.")

    for indicator in search_terms:
        print(f"\nSearching for: {indicator}")
    
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

        # Extract 'partners' directly
        partners = recent_tags.loc[recent_tags['summary'] == indicator, 'partners'].values
        observed_by = partners[0].replace("{{", "").replace("}}", "") if len(partners) > 0 else "N/A"

        # === VIRUSTOTAL ===
        vt_url = f"https://www.virustotal.com/gui/ip-address/{indicator}"
        vt_data = fetch_virustotal_data(indicator)

        if vt_data:
            attributes = vt_data.get("data", {}).get("attributes", {})
            services = attributes.get("services", [])
            ports = [s.get("port") for s in services if "port" in s]
            vt_entry = {
                'search_term': indicator,
                'timestamp': timestamp,
                'title': f"VirusTotal API Report for {indicator}",
                'link': vt_url,
                'asn': attributes.get('asn'),
                'as_owner': attributes.get('as_owner'),
                'country': attributes.get('country'),
                'network': attributes.get('network'),
                'last_analysis_stats': attributes.get('last_analysis_stats'),
                'reputation': attributes.get('reputation'),
                'total_votes': attributes.get('total_votes'),
                'open_ports': ports,
                'publish_date': "N/A"
            }

            if vt_url not in saved_links:
                save_to_json(vt_entry, FILE_PATH)
                saved_links.add(vt_url)
                print(f"Saved VT report for {indicator}.")
            else:
                print(f"VirusTotal entry for {indicator} already saved.")

        # === OTX ===
        otx_url = f"https://otx.alienvault.com/indicator/ip/{indicator}"
        otx_data = fetch_otx_data(indicator)
    
        if otx_data:
            otx_entry = {
                'search_term': indicator,
                'timestamp': timestamp,
                'title': f"OTX API Report for {indicator}",
                'link': otx_url,
                'base_indicator': otx_data.get('base_indicator', {}),
                'reputation': otx_data.get('reputation'),
                'geo': otx_data.get('geo', {}),
                'whois': otx_data.get('whois', {}),
                'open_ports': otx_data.get('open_ports', []),
                'publish_date': "N/A"
            }

            if otx_url not in saved_links:
                save_to_json(otx_entry, FILE_PATH)
                saved_links.add(otx_url)
                print(f"Saved OTX report for {indicator}.")
            else:
                print(f"OTX entry for {indicator} already saved.")

if __name__ == "__main__":
    main()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jaskew\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Found 3 unique search terms in 'summary' column.
Loaded 0 previously saved links.

Searching for: 190.92.174.130
Saved VT report for 190.92.174.130.
Saved OTX report for 190.92.174.130.

Searching for: 190.92.174.30
Saved VT report for 190.92.174.30.
Saved OTX report for 190.92.174.30.

Searching for: 190.92.174.36
Saved VT report for 190.92.174.36.
Saved OTX report for 190.92.174.36.


In [ ]:
import os
import json
from docx import Document

# File paths
FILE_PATH = r'C:\Users\jaskew\Documents\project_repository\data\processed\BingSearchData.json'
TEMPLATE_PATH = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\I&W Report Template.docx"
OUTPUT_DIR = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports"

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_data(file_path):
    """ Load raw data from JSON file. """
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return []

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return []
    except IOError as e:
        print(f"Error reading file: {e}")
        return []

def consolidate_data(data):
    """ Consolidate VT and OTX data for each search term. """
    consolidated = {}

    for entry in data:
        search_term = entry['search_term']

        if search_term not in consolidated:
            # Initialize a new record structure
            consolidated[search_term] = {
                "search_term": search_term,
                "timestamp": entry['timestamp'],
                "vt_link": "",
                "otx_link": "",
                "asn": entry.get('asn', 'N/A'),
                "as_owner": entry.get('as_owner', 'N/A'),
                "country": entry.get('country', 'N/A'),
                "network": entry.get('network', 'N/A'),
                "last_analysis_stats": {},
                "reputation": entry.get('reputation', 'N/A'),
                "total_votes": {},
                "open_ports": [],
                "whois": "",
                "base_indicator": {}
            }

        # Populate VirusTotal data
        if "VirusTotal API Report" in entry['title']:
            consolidated[search_term].update({
                "vt_link": entry['link'],
                "last_analysis_stats": entry.get('last_analysis_stats', {}),
                "total_votes": entry.get('total_votes', {}),
                "open_ports": entry.get('open_ports', [])
            })

        # Populate OTX data
        elif "OTX API Report" in entry['title']:
            consolidated[search_term].update({
                "otx_link": entry['link'],
                "whois": entry.get('whois', ''),
                "base_indicator": entry.get('base_indicator', {}),
                "reputation": entry.get('reputation', 'N/A')
            })

    # Convert to list format for easier iteration
    return list(consolidated.values())

def generate_replacements(entry):
    """ Generate a dictionary of replacements based on consolidated JSON entry. """
    return {
        "{{title}}": f"I&W Report for {entry['search_term']}",
        "{{ip_address}}": entry.get('search_term', 'N/A'),
        "{{indicator_types}}": "IPv4 Address" if '.' in entry.get('search_term', '') else "Domain",
        "{{date_observed}}": entry.get('timestamp', 'N/A'),
        "{{observed_by}}": observed_by,
        "{{sources}}": f"{entry.get('vt_link', 'N/A')}, {entry.get('otx_link', 'N/A')}",
        "{{asn_number}}": entry.get('asn', 'N/A'),
        "{{as_owner}}": entry.get('as_owner', 'N/A')
        #"{{reputation}}": entry.get('reputation', 'N/A'),
        #"{{open_ports}}": ', '.join(map(str, entry.get('open_ports', []))),
        #"{{whois}}": entry.get('whois', 'N/A'),
        #"{{malicious_count}}": entry.get('last_analysis_stats', {}).get('malicious', 'N/A'),
        #"{{suspicious_count}}": entry.get('last_analysis_stats', {}).get('suspicious', 'N/A'),
        #"{{undetected_count}}": entry.get('last_analysis_stats', {}).get('undetected', 'N/A')
    }

def fill_word_template(template_path, output_path, replacements):
    """ Fills placeholders in the Word document with provided replacements. """
    doc = Document(template_path)

    # Replace placeholders in paragraphs
    for para in doc.paragraphs:
        for key, value in replacements.items():
            if key in para.text:
                para.text = para.text.replace(key, str(value))

    # Replace placeholders in tables
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for key, value in replacements.items():
                    if key in cell.text:
                        cell.text = cell.text.replace(key, str(value))

    # Save the modified document
    doc.save(output_path)

def generate_reports(template_path, output_dir, data):
    """ Generate reports for each consolidated record. """
    for index, entry in enumerate(data):
        replacements = generate_replacements(entry)
        output_path = os.path.join(output_dir, f"Report_{index + 1}_{entry['search_term']}.docx")
        fill_word_template(template_path, output_path, replacements)
        print(f"Generated report for {entry['search_term']} at {output_path}")

def main():
    # Load raw data
    raw_data = load_data(FILE_PATH)

    # Consolidate data for reporting
    consolidated_data = consolidate_data(raw_data)

    if consolidated_data:
        print(f"Loaded and consolidated {len(consolidated_data)} records.")
        generate_reports(TEMPLATE_PATH, OUTPUT_DIR, consolidated_data)
    else:
        print("No valid data found for report generation.")

if __name__ == "__main__":
    main()


Loaded and consolidated 3 records.
Generated report for 190.92.174.130 at C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\Report_1_190.92.174.130.docx
Generated report for 190.92.174.30 at C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\Report_2_190.92.174.30.docx
Generated report for 190.92.174.36 at C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\Report_3_190.92.174.36.docx
